# Internet Connectivity Prediction — Data Extraction & Exploration
**Runtime:** Google Colab (CPU — no TPU needed for this notebook)

## Workflow
```
Step 1: Download Ookla mobile tiles for all of India
Step 2: Build full candidate patch grid over India (~170K patches)
Step 3: Batch-extract stratification features (NTL, built-up, elevation) via GEE export
Step 4: Spatial join — assign Ookla labels to patches
Step 5: Classify label confidence (positive / negative / ambiguous)
Step 6: Stratified sampling → ~7,000 patches
Step 7: Export HLS imagery ONLY for sampled patches (224×224×6 bands)
```

## 0. Setup

In [ ]:
import os

REPO = 'satellite-internet-prediction'

# Only cd if not already inside the repo (prevents nesting on re-runs)
if not os.getcwd().endswith(REPO):
    if os.path.exists(REPO):
        %cd {REPO}
    else:
        !git clone https://github.com/tatyana21111/{REPO}.git
        %cd {REPO}

# Pull latest changes
!git pull

In [ ]:
!pip install -q earthengine-api geopandas pyarrow pandas matplotlib seaborn shapely requests

In [ ]:
import ee
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from shapely.geometry import Point, box

ee.Authenticate()
ee.Initialize(
    project='ziyi-ml-internet-connectivity', 
    opt_url='https://earthengine-highvolume.googleapis.com'
)

Path('data/raw').mkdir(parents=True, exist_ok=True)
Path('data/interim').mkdir(parents=True, exist_ok=True)
Path('data/processed').mkdir(parents=True, exist_ok=True)
Path('outputs/figures').mkdir(parents=True, exist_ok=True)

In [ ]:
import random

random.seed(42)
np.random.seed(42)

## Step 1: Download Ookla Mobile Tiles for All of India

In [ ]:
# India bounding box
INDIA_BBOX = [68.0, 6.5, 97.5, 37.5]  # [minLon, minLat, maxLon, maxLat]
india_geom = ee.Geometry.BBox(*INDIA_BBOX)

# Quarter → date mapping
QUARTER_DATES = {1: '01-01', 2: '04-01', 3: '07-01', 4: '10-01'}
YEAR    = 2019
QUARTER = 1
date_str = f'{YEAR}-{QUARTER_DATES[QUARTER]}'

ookla_asset = (
    f'projects/sat-io/open-datasets/network/mobile_tiles/'
    f'{date_str}_performance_mobile_tiles'
)
print(f'Loading: {ookla_asset}')

ookla_fc = ee.FeatureCollection(ookla_asset).filterBounds(india_geom)
print(f'Ookla tiles in India: {ookla_fc.size().getInfo()}')

In [ ]:
# Export Ookla tiles to Drive (avoids getInfo() memory limit for large FCs)
task_ookla = ee.batch.Export.table.toDrive(
    collection=ookla_fc,
    description=f'ookla_india_{YEAR}_Q{QUARTER}',
    fileFormat='CSV',
    folder='gee_exports'
)
task_ookla.start()
print('Ookla export task started — check GEE Tasks panel.')
print('Once complete, download CSV from Google Drive to data/raw/')

## Step 1b: Copy Ookla CSV from Google Drive to local data/raw/
# Run AFTER the GEE export task in Step 1 has completed.

In [ ]:
from google.colab import drive
import shutil

drive.mount('/content/drive', force_remount=False)

DRIVE_FOLDER  = '/content/drive/MyDrive/gee_exports'
ookla_fname   = f'ookla_india_{YEAR}_Q{QUARTER}.csv'
ookla_src     = f'{DRIVE_FOLDER}/{ookla_fname}'
ookla_dst     = f'data/raw/{ookla_fname}'

if not Path(ookla_dst).exists():
    shutil.copy(ookla_src, ookla_dst)
    print(f'Copied {ookla_fname} → {ookla_dst}')
else:
    print(f'Already exists: {ookla_dst}')

## Step 2: Build Full Candidate Patch Grid over India

Generate a 6.72 km grid matching **Prithvi-300M's 224 × 224 px @ 30 m** native resolution (224 × 30 m = 6 720 m per side → 0.0604° at Indian latitudes). Patches are clipped to the India boundary so only land-interior centres are kept (~170 K patches expected).

In [ ]:
from shapely.geometry import Point
import itertools, zipfile, io, requests as _req

# Download India boundary from Natural Earth if not already present
_boundary_path = Path('data/raw/india_boundary.gpkg')
if not _boundary_path.exists():
    print('Downloading India boundary from Natural Earth …')
    url = 'https://naturalearth.s3.amazonaws.com/50m_cultural/ne_50m_admin_0_countries.zip'
    z = zipfile.ZipFile(io.BytesIO(_req.get(url).content))
    z.extractall('data/raw/_ne_tmp')
    world = gpd.read_file('data/raw/_ne_tmp/ne_50m_admin_0_countries.shp')
    india_boundary = world[world['ADMIN'] == 'India'][['geometry']].reset_index(drop=True)
    india_boundary.to_file(_boundary_path, driver='GPKG')
    print('Saved: data/raw/india_boundary.gpkg')
else:
    print('Using cached: data/raw/india_boundary.gpkg')

india_boundary = gpd.read_file(_boundary_path)

# Generate 6.72 km grid matching Prithvi 224×224 @ 30 m resolution
# 224 px × 30 m = 6 720 m;  6.72 km / 111.32 km per degree ≈ 0.0604°
PATCH_SIZE_DEG = 6.72 / 111.32   # ~0.0604 degrees at Indian latitudes
BUFFER_M       = 3360             # half of 6 720 m patch side

bounds = india_boundary.total_bounds  # [minx, miny, maxx, maxy]

xs = np.arange(bounds[0], bounds[2], PATCH_SIZE_DEG)
ys = np.arange(bounds[1], bounds[3], PATCH_SIZE_DEG)

grid_points = [Point(x, y) for x, y in itertools.product(xs, ys)]
grid_gdf = gpd.GeoDataFrame(geometry=grid_points, crs='EPSG:4326')

# Keep only patches whose centre falls within India (drops ocean / neighbours)
grid_gdf = gpd.sjoin(grid_gdf, india_boundary[['geometry']], predicate='within')
grid_gdf = grid_gdf.drop(columns=['index_right']).reset_index(drop=True)

grid_gdf['patch_id'] = range(len(grid_gdf))
grid_gdf['lon']      = grid_gdf.geometry.x
grid_gdf['lat']      = grid_gdf.geometry.y

print(f'Total patches covering India: {len(grid_gdf):,}')
# Expected: ~170,000
grid_gdf.to_file('data/raw/india_patch_grid.gpkg', driver='GPKG')
print('Saved: data/raw/india_patch_grid.gpkg')

## Step 3: Batch-Extract Stratification Features for ALL Patches
Extract NTL (VIIRS nightlights), built-up density (GHSL), and elevation (SRTM) for every patch using a single GEE batch export — much faster than calling `getInfo()` per patch.

In [ ]:
# Build combined image for stratification
viirs = (
    ee.ImageCollection('NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG')
    .filterDate('2019-01-01', '2019-12-31')
    .select('avg_rad')
    .median()
    .rename('ntl')
)
ghsl = ee.Image('JRC/GHSL/P2023A/GHS_BUILT_S/2020').select('built_surface').rename('builtup')
srtm = ee.Image('USGS/SRTMGL1_003').select('elevation').rename('elevation')

combined_image = viirs.addBands(ghsl).addBands(srtm)

# Build EE FeatureCollection from a DataFrame chunk
CHUNK_SIZE = 5000  # stay well under GEE's 10 MB request payload limit

def build_ee_fc(df_chunk):
    features = [
        ee.Feature(
            ee.Geometry.Point([row['lon'], row['lat']]),
            {'patch_id': row['patch_id']}
        )
        for _, row in df_chunk.iterrows()
    ]
    return ee.FeatureCollection(features)

def extract_features(feature):
    region = feature.geometry().buffer(BUFFER_M).bounds()
    stats  = combined_image.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=region,
        scale=500,
        maxPixels=1e6,
    )
    return feature.set(stats)

# Split grid into chunks and launch one export task per chunk.
# Sending all ~170 K features at once exceeds GEE's 10 MB payload limit.
chunks = [grid_gdf.iloc[i:i + CHUNK_SIZE] for i in range(0, len(grid_gdf), CHUNK_SIZE)]
print(f'Launching {len(chunks)} export tasks ({CHUNK_SIZE} patches each) …')

tasks = []
for i, chunk in enumerate(chunks):
    fc_chunk   = build_ee_fc(chunk)
    enriched   = fc_chunk.map(extract_features)
    task = ee.batch.Export.table.toDrive(
        collection=enriched,
        description=f'india_patch_strat_chunk_{i:03d}',
        fileFormat='CSV',
        folder='gee_exports'
    )
    task.start()
    tasks.append(task)
    if i % 5 == 0:
        print(f'  started chunk {i+1}/{len(chunks)}')

print('All stratification export tasks started — check GEE Tasks panel.')
print('Once all are complete, download the CSVs from Drive and merge them:')
print('  pd.concat([pd.read_csv(f) for f in sorted(Path("data/interim").glob("india_patch_strat_chunk_*.csv"))])')

## Step 3b: Merge GEE chunk CSVs into one file
# Run AFTER all GEE export tasks in Step 3 have finished (check the GEE Tasks panel).
# The chunk CSVs land in Google Drive → gee_exports/ as india_patch_strat_chunk_*.csv

In [ ]:
from google.colab import drive
import glob, shutil

# Mount Google Drive (will prompt for auth once per session)
drive.mount('/content/drive', force_remount=False)

DRIVE_FOLDER = '/content/drive/MyDrive/gee_exports'
OUT_CSV      = 'data/interim/india_patch_stratification_features.csv'

# Copy chunk CSVs from Drive to local interim folder
chunk_files_drive = sorted(glob.glob(f'{DRIVE_FOLDER}/india_patch_strat_chunk_*.csv'))
print(f'Found {len(chunk_files_drive)} chunk file(s) in Drive.')

for src in chunk_files_drive:
    shutil.copy(src, 'data/interim/')

# Merge all chunks into one file
chunk_files_local = sorted(glob.glob('data/interim/india_patch_strat_chunk_*.csv'))
strat_df = pd.concat([pd.read_csv(f) for f in chunk_files_local], ignore_index=True)
strat_df.to_csv(OUT_CSV, index=False)
print(f'Merged {len(chunk_files_local)} chunks → {len(strat_df):,} rows saved to {OUT_CSV}')

## Step 4: Spatial Join — Assign Ookla Labels to Patches

In [ ]:
import json

# Load patch stratification features
features_df = pd.read_csv('data/interim/india_patch_stratification_features.csv')

# Parse lon/lat from GEE's .geo column: {"type":"Point","coordinates":[lon, lat]}
features_df['lon'] = features_df['.geo'].apply(lambda x: json.loads(x)['coordinates'][0])
features_df['lat'] = features_df['.geo'].apply(lambda x: json.loads(x)['coordinates'][1])

# Load Ookla data
ookla_df = pd.read_csv(f'data/raw/ookla_india_{YEAR}_Q{QUARTER}.csv')
ookla_df['geometry'] = ookla_df['.geo'].apply(
    lambda x: gpd.GeoDataFrame.from_features(
        [{'type': 'Feature', 'geometry': json.loads(x), 'properties': {}}]
    ).geometry[0]
)
ookla_gdf = gpd.GeoDataFrame(ookla_df, geometry='geometry', crs='EPSG:4326')

# Build patch polygons (6.72 km × 6.72 km boxes — matches Prithvi 224×224 @ 30 m)
half = PATCH_SIZE_DEG / 2
patch_gdf = gpd.GeoDataFrame(
    features_df,
    geometry=[
        box(lon - half, lat - half, lon + half, lat + half)
        for lon, lat in zip(features_df['lon'], features_df['lat'])
    ],
    crs='EPSG:4326'
)

# Spatial join: count Ookla tests per patch
joined = gpd.sjoin(
    ookla_gdf[['avg_d_kbps', 'avg_u_kbps', 'avg_lat_ms', 'tests', 'geometry']],
    patch_gdf[['patch_id', 'geometry']], how='right', predicate='within'
)

ookla_per_patch = joined.groupby('patch_id').agg(
    ookla_tests=('tests',      'sum'),
    avg_d_kbps =('avg_d_kbps', 'mean'),
    avg_u_kbps =('avg_u_kbps', 'mean'),
    avg_lat_ms =('avg_lat_ms', 'mean'),
).reset_index()

patch_gdf = patch_gdf.merge(ookla_per_patch, on='patch_id', how='left')
patch_gdf['ookla_tests'] = patch_gdf['ookla_tests'].fillna(0)
print(patch_gdf.shape)
patch_gdf.head()

## Step 5: Classify Label Confidence

In [ ]:
def classify_confidence(row):
    """Assign label confidence based on Ookla test count and NTL."""
    if row['ookla_tests'] >= 5:
        return 'confirmed_positive'   # well-tested connectivity
    elif row['ookla_tests'] == 0 and row.get('ntl', 0) < 0.5:
        return 'confirmed_negative'   # dark area, no tests
    else:
        return 'ambiguous'            # low test count or lit but no tests

patch_gdf['label_confidence'] = patch_gdf.apply(classify_confidence, axis=1)
print(patch_gdf['label_confidence'].value_counts())

patch_gdf.drop(columns='geometry').to_csv('data/interim/patches_labelled.csv', index=False)
print('Saved: data/interim/patches_labelled.csv')

## Step 6: Stratified Sampling → ~7,000 Patches

In [ ]:
# Stratify by label_confidence × NTL quartile
confident = patch_gdf[patch_gdf['label_confidence'] != 'ambiguous'].copy()
confident['ntl_quartile'] = pd.qcut(confident['ntl'].fillna(0), q=4, labels=['Q1','Q2','Q3','Q4'])
confident['stratum'] = confident['label_confidence'] + '_' + confident['ntl_quartile'].astype(str)

TARGET_N = 7000
stratum_counts = confident['stratum'].value_counts()
stratum_fracs  = (stratum_counts / stratum_counts.sum() * TARGET_N).round().astype(int)

sampled = pd.concat([
    confident[confident['stratum'] == stratum].sample(
        n=min(n, len(confident[confident['stratum'] == stratum])),
        random_state=42
    )
    for stratum, n in stratum_fracs.items()
])

print(f'Sampled patches: {len(sampled):,}')
print(sampled['stratum'].value_counts())

sampled.drop(columns='geometry', errors='ignore').to_csv('data/processed/sampled_patches.csv', index=False)
print('Saved: data/processed/sampled_patches.csv')

## Step 7: Export HLS Imagery for Sampled Patches Only
This is the expensive step — only runs on ~7,000 patches, not the full ~170K grid.

In [ ]:
BANDS_S30 = ['B2', 'B3', 'B4', 'B8A', 'B11', 'B12']   # HLS S30 band names (no leading zero)
BANDS_L30 = ['B2', 'B3', 'B4', 'B5',  'B6',  'B7' ]   # HLS L30 band names
RENAMED   = ['Blue', 'Green', 'Red', 'NIR_Narrow', 'SWIR1', 'SWIR2']

def mask_hls_clouds(image):
    fmask  = image.select('Fmask')
    cloud  = fmask.bitwiseAnd(1 << 1).eq(0)
    shadow = fmask.bitwiseAnd(1 << 3).eq(0)
    return image.updateMask(cloud.And(shadow))

def build_hls_composite(start_date, end_date, region):
    s30 = (
        ee.ImageCollection('NASA/HLS/HLSS30/v002')
        .filterBounds(region).filterDate(start_date, end_date)
        .filter(ee.Filter.lt('CLOUD_COVERAGE', 30))
        .map(mask_hls_clouds).select(BANDS_S30, RENAMED)
    )
    l30 = (
        ee.ImageCollection('NASA/HLS/HLSL30/v002')
        .filterBounds(region).filterDate(start_date, end_date)
        .filter(ee.Filter.lt('CLOUD_COVERAGE', 30))
        .map(mask_hls_clouds).select(BANDS_L30, RENAMED)
    )
    return s30.merge(l30).median()

india_region   = ee.Geometry.BBox(*INDIA_BBOX)
composite_2019 = build_hls_composite('2019-01-01', '2019-12-31', india_region)
print('HLS composite built.')

In [ ]:
import time

def export_patch(row, composite, out_dir):
    """Download a single 224×224 patch as GeoTIFF, skipping if already done."""
    out_path = Path(f"{out_dir}/{row['patch_id']}.tif")
    if out_path.exists():
        return 'skipped'
    point  = ee.Geometry.Point([row['lon'], row['lat']])
    region = point.buffer(BUFFER_M).bounds()
    for attempt in range(5):
        try:
            url = composite.getDownloadURL({
                'region':     region,
                'dimensions': '224x224',
                'format':     'GEO_TIFF',
                'bands':      RENAMED,
            })
            r = requests.get(url, stream=True, timeout=120)
            r.raise_for_status()
            Path(out_dir).mkdir(parents=True, exist_ok=True)
            with open(out_path, 'wb') as f:
                f.write(r.content)
            return 'ok'
        except Exception as e:
            if attempt < 4:
                time.sleep(2 ** attempt)   # exponential back-off
            else:
                print(f"FAILED {row['patch_id']}: {e}")
                return 'failed'

OUT_DIR = 'data/raw/patches_2019'
results = {'ok': 0, 'skipped': 0, 'failed': 0}

# max_workers=4 keeps request rate within GEE thumbnail limits
with ThreadPoolExecutor(max_workers=4) as executor:
    futures = {executor.submit(export_patch, row, composite_2019, OUT_DIR): row['patch_id']
               for _, row in sampled.iterrows()}
    for i, future in enumerate(futures):
        results[future.result()] += 1
        if (i + 1) % 100 == 0:
            print(f"  {i+1}/{len(sampled)} — {results}")

print(f'Done: {results}')

## Step 8: Build Aggregate Targets & Train/Val/Test Split

Compute `coverage_fraction` and `log_mean_tests` from the Ookla spatial join,
assign geographic train/val/test splits, and save a single CSV that all
downstream notebooks (NB02-07) can load directly — eliminating repeated
Ookla parsing, spatial joins, and split logic.

In [ ]:
import json
from sklearn.model_selection import train_test_split

# ── Load sampled patches ─────────────────────────────────────
sampled = pd.read_csv('data/processed/sampled_patches.csv')
available = set(f.stem for f in Path('data/raw/patches_2019').glob('*.tif'))
sampled = sampled[sampled['patch_id'].isin(available)].reset_index(drop=True)
sampled['has_coverage'] = (sampled['label_confidence'] == 'confirmed_positive').astype(int)
print(f'Patches available: {len(sampled):,}')

# ── Load Ookla sub-tile CSV ──────────────────────────────────
ookla_path = Path('data/raw/ookla_india_2019_Q1.csv')
if not ookla_path.exists():
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    import shutil
    shutil.copy('/content/drive/MyDrive/gee_exports/ookla_india_2019_Q1.csv', ookla_path)

ookla_raw = pd.read_csv(ookla_path)
ookla_raw['geometry'] = ookla_raw['.geo'].apply(
    lambda x: gpd.GeoDataFrame.from_features(
        [{'type': 'Feature', 'geometry': json.loads(x), 'properties': {}}]
    ).geometry[0]
)
ookla_gdf = gpd.GeoDataFrame(ookla_raw, geometry='geometry', crs='EPSG:4326')
print(f'Ookla tiles: {len(ookla_gdf):,}')

# ── Spatial join: count Ookla sub-tiles per patch ────────────
PATCH_SIZE_M   = 6720.0
_PATCH_DEG     = PATCH_SIZE_M / 111_320.0
_half          = _PATCH_DEG / 2
EARTH_CIRC_M   = 40_075_016.686
ZOOM16_SIDE_EQ = EARTH_CIRC_M / (2 ** 16)

patch_gdf = gpd.GeoDataFrame(
    sampled[['patch_id', 'lon', 'lat']],
    geometry=[box(lon - _half, lat - _half, lon + _half, lat + _half)
              for lon, lat in zip(sampled['lon'], sampled['lat'])],
    crs='EPSG:4326'
)

joined = gpd.sjoin(
    ookla_gdf[['tests', 'geometry']],
    patch_gdf[['patch_id', 'geometry']],
    how='inner', predicate='intersects',
)

ookla_per_patch = (
    joined.groupby('patch_id')
    .agg(total_tests=('tests', 'sum'),
         n_subtiles=('tests', 'count'),
         mean_tests=('tests', 'mean'))
    .reset_index()
)

sampled = sampled.merge(ookla_per_patch, on='patch_id', how='left')
for col in ['total_tests', 'n_subtiles', 'mean_tests']:
    sampled[col].fillna(0, inplace=True)

# ── Aggregate targets ────────────────────────────────────────
def total_possible_tiles(lat):
    tile_side = ZOOM16_SIDE_EQ * np.cos(np.radians(lat))
    return (PATCH_SIZE_M / tile_side) ** 2

sampled['total_possible_tiles'] = sampled['lat'].apply(total_possible_tiles)
sampled['coverage_fraction'] = (sampled['n_subtiles'] / sampled['total_possible_tiles']).clip(0, 1)
sampled['log_mean_tests'] = np.log1p(sampled['mean_tests'])

print(f'coverage_fraction  — mean: {sampled["coverage_fraction"].mean():.4f}')
print(f'log_mean_tests     — mean: {sampled["log_mean_tests"].mean():.4f}')

# ── Geographic train/val/test split ──────────────────────────
northeast_mask = (sampled['lon'] > 89.0) & (sampled['lat'] > 21.0)
sampled['split'] = 'train'
sampled.loc[northeast_mask, 'split'] = 'test'

train_val_idx = sampled[~northeast_mask].index
train_idx, val_idx = train_test_split(
    train_val_idx, test_size=0.2,
    stratify=sampled.loc[train_val_idx, 'has_coverage'],
    random_state=42
)
sampled.loc[val_idx, 'split'] = 'val'

print(f'\nSplit: {sampled["split"].value_counts().to_dict()}')

# ── Save ─────────────────────────────────────────────────────
OUT_PATH = 'data/processed/patches_with_splits.csv'
keep_cols = [
    'patch_id', 'lon', 'lat', 'label_confidence', 'has_coverage',
    'ntl', 'builtup', 'elevation',
    'total_tests', 'n_subtiles', 'mean_tests',
    'coverage_fraction', 'log_mean_tests',
    'split',
]
sampled[keep_cols].to_csv(OUT_PATH, index=False)
print(f'Saved: {OUT_PATH}  ({len(sampled):,} rows, {len(keep_cols)} columns)')

In [ ]:
!git config --global user.email "tatyana211zy@outlook.com"
!git config --global user.name "tatyana21111"

# Force-add only the small processed CSVs (data/ is gitignored for large files)
!git add -f data/processed/sampled_patches.csv
!git add -f data/processed/patches_with_splits.csv
!git add outputs/

!git diff --cached --quiet || git commit -m "NB01: Add sampled patches, aggregate targets, and train/val/test splits"
!git push